In [1]:
import pandas as pd

train_df = pd.read_csv(r"C:\AIML Projects\Mental_health_survey\data\train.csv") 

print("Shape of the dataset:", train_df.shape)
print("\nColumn Names:\n", train_df.columns.tolist())

train_df.head()


Shape of the dataset: (140700, 20)

Column Names:
 ['id', 'Name', 'Gender', 'Age', 'City', 'Working Professional or Student', 'Profession', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness', 'Depression']


,id,Name,Gender,Age,City,Working Professional or Student,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,0,Aaradhya,Female,49.0,Ludhiana,Working Professional,Chef,NaN,5.0,NaN,NaN,2.0,More than 8 hours,Healthy,BHM,No,1.0,2.0,No,0
1,1,Vivan,Male,26.0,Varanasi,Working Professional,Teacher,NaN,4.0,NaN,NaN,3.0,Less than 5 hours,Unhealthy,LLB,Yes,7.0,3.0,No,1
2,2,Yuvraj,Male,33.0,Visakhapatnam,Student,NaN,5.0,NaN,8.97,2.0,NaN,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
3,3,Yuvraj,Male,22.0,Mumbai,Working Professional,Teacher,NaN,5.0,NaN,NaN,1.0,Less than 5 hours,Moderate,BBA,Yes,10.0,1.0,Yes,1
4,4,Rhea,Female,30.0,Kanpur,Working Professional,Business Analyst,NaN,1.0,NaN,NaN,1.0,5-6 hours,Unhealthy,BBA,Yes,9.0,4.0,Yes,0


In [2]:
train_df.drop(columns=["id", "Name", "City"], inplace=True)

missing_summary = train_df.isnull().sum()

print("Columns after dropping:", train_df.columns.tolist())
print("\nMissing values in each column:\n", missing_summary[missing_summary > 0])


Columns after dropping: ['Gender', 'Age', 'Working Professional or Student', 'Profession', 'Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Sleep Duration', 'Dietary Habits', 'Degree', 'Have you ever had suicidal thoughts ?', 'Work/Study Hours', 'Financial Stress', 'Family History of Mental Illness', 'Depression']

Missing values in each column:
 Profession             36630
Academic Pressure     112803
Work Pressure          27918
CGPA                  112802
Study Satisfaction    112803
Job Satisfaction       27910
Dietary Habits             4
Degree                     2
Financial Stress           4
dtype: int64


In [3]:
for col in ['Profession', 'Dietary Habits', 'Degree']:
    train_df[col].fillna(train_df[col].mode()[0], inplace=True)

for col in ['Academic Pressure', 'Work Pressure', 'CGPA', 'Study Satisfaction', 'Job Satisfaction', 'Financial Stress']:
    train_df[col].fillna(train_df[col].median(), inplace=True)

print("Missing values left:", train_df.isnull().sum().sum())


Missing values left: 0


C:\Users\Gopinath\AppData\Local\Temp\ipykernel_3968\464233663.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df[col].fillna(train_df[col].mode()[0], inplace=True)
C:\Users\Gopinath\AppData\Local\Temp\ipykernel_3968\464233663.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

cat_cols = [
    'Gender', 'Working Professional or Student', 'Profession',
    'Dietary Habits', 'Degree', 'Sleep Duration',
    'Have you ever had suicidal thoughts ?', 'Family History of Mental Illness'
]

for col in cat_cols:
    train_df[col] = le.fit_transform(train_df[col])

train_df[cat_cols].head()


,Gender,Working Professional or Student,Profession,Dietary Habits,Degree,Sleep Duration,Have you ever had suicidal thoughts ?,Family History of Mental Illness
0,0,1,10,7,33,29,0,0
1,1,1,55,20,63,27,1,0
2,1,0,55,7,21,15,1,0
3,1,1,55,15,28,27,1,1
4,0,1,9,20,28,15,1,1


In [5]:
from sklearn.preprocessing import StandardScaler

num_cols = [
    'Age', 'Academic Pressure', 'Work Pressure', 'CGPA',
    'Study Satisfaction', 'Job Satisfaction', 'Work/Study Hours', 'Financial Stress'
]

scaler = StandardScaler()

train_df[num_cols] = scaler.fit_transform(train_df[num_cols])

train_df[num_cols].head()


,Age,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Work/Study Hours,Financial Stress
0,0.695360,-0.045698,1.589714,0.033784,0.018013,-0.772518,-1.363057,-0.699617
1,-1.161867,-0.045698,0.795176,0.033784,0.018013,0.016183,0.193928,0.007793
2,-0.596624,3.194278,0.000638,1.869755,-1.632006,0.016183,-0.844062,-1.407027
3,-1.484863,-0.045698,1.589714,0.033784,0.018013,-1.561219,0.972421,-1.407027
4,-0.838871,-0.045698,-1.588437,0.033784,0.018013,-1.561219,0.712923,0.715203


In [6]:
from sklearn.model_selection import train_test_split

X = train_df.drop(columns=["Depression"])
y = train_df["Depression"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)


X_train: (112560, 16)
X_val: (28140, 16)
y_train: (112560,)
y_val: (28140,)


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

class MentalHealthDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = MentalHealthDataset(X_train, y_train)
val_dataset = MentalHealthDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [8]:
class DepressionPredictor(nn.Module):
    def __init__(self, input_dim):
        super(DepressionPredictor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
            nn.Sigmoid()  
        )

    def forward(self, x):
        return self.net(x)


input_dim = X_train.shape[1]
model = DepressionPredictor(input_dim)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [9]:
epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}")


Epoch 1/10 - Train Loss: 0.2356 - Val Loss: 0.1650
Epoch 2/10 - Train Loss: 0.1771 - Val Loss: 0.1647
Epoch 3/10 - Train Loss: 0.1721 - Val Loss: 0.1637
Epoch 4/10 - Train Loss: 0.1698 - Val Loss: 0.1604
Epoch 5/10 - Train Loss: 0.1693 - Val Loss: 0.1685
Epoch 6/10 - Train Loss: 0.1683 - Val Loss: 0.1643
Epoch 7/10 - Train Loss: 0.1680 - Val Loss: 0.1624
Epoch 8/10 - Train Loss: 0.1667 - Val Loss: 0.1598
Epoch 9/10 - Train Loss: 0.1666 - Val Loss: 0.1608
Epoch 10/10 - Train Loss: 0.1670 - Val Loss: 0.1625


In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()
y_preds = []
y_true = []

with torch.no_grad():
    for inputs, labels in val_loader:
        outputs = model(inputs).squeeze()
        preds = (outputs >= 0.5).int()
        y_preds.extend(preds.tolist())
        y_true.extend(labels.tolist())

y_preds = torch.tensor(y_preds)
y_true = torch.tensor(y_true)

accuracy = accuracy_score(y_true, y_preds)
precision = precision_score(y_true, y_preds)
recall = recall_score(y_true, y_preds)
f1 = f1_score(y_true, y_preds)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


Accuracy: 0.9371
Precision: 0.8368
Recall: 0.8124
F1 Score: 0.8245


In [13]:
import os

os.makedirs('model', exist_ok=True)

torch.save(model.state_dict(), 'model/depression_model.pth')

print("Model saved to model/depression_model.pth")


Model saved to model/depression_model.pth
